# Test03: doctor prediction and senior-review feedback

For each held-out test note, a doctor-profile LLM proposes three distinct ICD-11 codes, ranked best first. Python checks them against MMS and the enriched CSV. One judge call reviews all three together, including the title, definition, and text of every CSV entry sharing the first four characters of each code's stem. Duplicate families are included only once. If the judge rejects the ranking, its feedback is sent to the doctor for another attempt.

There is one initial attempt plus up to three retries. Incorrect counts, duplicates, invalid MMS codes, and codes absent from the CSV are retried without a judge call. If no attempt is approved, the last available MMS-valid, CSV-backed codes are returned **unapproved**. Fewer than three are marked **incomplete**; if no eligible codes are available, the prediction is empty.

**Research only:** both roles are model outputs, not licensed doctors; agreement is not clinical validation. These 10 notes have already been used in earlier experiments, so reserve a separate unseen set for a final evaluation. Running the prediction cell sends the notes and full matching ICD code-family descriptions to the model provider and may incur charges; the expanded context can increase cost and latency. It can make up to eight model calls per note and overwrites the Test03 output files in `out/`. Saved outputs from earlier runs do not include the expanded context; rerun all cells to use it.

In [1]:
import importlib
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
import condition_utils
import test03_review
from score import load_valid
importlib.reload(test03_review)

root = Path.cwd()
if not (root / 'llm.py').is_file() or not (root / 'score.py').is_file():
    raise FileNotFoundError('Start the notebook with the project root as its working directory.')
provider = 'openai'
doctor_model = None  # None uses the provider's MODEL environment variable or llm.py default
judge_model = None   # Set a model ID to use a different judge
max_retries = 3      # Three retries after the initial attempt
api_python = '/opt/homebrew/bin/python3'  # Python that can verify your network's TLS certificate
notes_path = root / 'data' / 'dev_notes_test.jsonl'
icd_path = root / 'data' / 'final-icd-codes.csv'
mms_path = root / 'data' / 'mms-2026-01.json'
doctor_prompt_path = root / 'prompts' / 'test03_doctor.txt'
judge_prompt_path = root / 'prompts' / 'test03_judge.txt'
predictions_path = root / 'out' / 'dev_predictions_test03.jsonl'
reviews_path = root / 'out' / 'dev_reviews_test03.jsonl'
per_note_path = root / 'out' / 'dev_per_note_test03.jsonl'

## Load notes, eligible codes, and prompts

Edit the doctor and judge prompt files to tune the experiment, then rerun this cell. The judge receives the ranked candidates and related CSV entries, not the gold label. Existing data files are read-only.

In [2]:
with notes_path.open(encoding='utf-8') as file:
    notes = [json.loads(line) for line in file if line.strip()]
if not notes:
    raise ValueError(f'{notes_path} contains no notes.')
for note in notes:
    if not isinstance(note, dict) or not isinstance(note.get('id'), str) or not isinstance(note.get('text'), str) or not isinstance(note.get('gold'), list) or not note['gold']:
        raise ValueError('Each note must have an id, text, and non-empty gold codes.')
if len({note['id'] for note in notes}) != len(notes):
    raise ValueError('Note IDs must be unique.')
if not isinstance(max_retries, int) or isinstance(max_retries, bool) or max_retries < 0:
    raise ValueError('max_retries must be a non-negative integer.')
catalog = test03_review.load_catalog(icd_path)
valid_codes = load_valid(mms_path)
eligible_codes = set(catalog) & valid_codes
if not eligible_codes:
    raise ValueError('No CSV codes match valid MMS categories.')
doctor_prompt = doctor_prompt_path.read_text(encoding='utf-8').strip()
judge_prompt = judge_prompt_path.read_text(encoding='utf-8').strip()
if not doctor_prompt or not judge_prompt:
    raise ValueError('Doctor and judge prompts must not be empty.')
print(f'Loaded {len(notes)} notes and {len(eligible_codes)} MMS-valid CSV codes.')
print(f'Maximum model calls: {len(notes) * 2 * (max_retries + 1)}')

Loaded 10 notes and 17979 MMS-valid CSV codes.
Maximum model calls: 80


## Generate and review predictions

Each completed prediction and its review history are flushed to separate JSONL files. If a call fails, the error is surfaced and completed notes remain on disk. Rerunning this cell starts over.

In [3]:
def call_doctor(prompt, context):
    print(f'  {current_note_id}: doctor request', flush=True)
    return condition_utils.call_model(prompt, context, root=root, provider=provider, model=doctor_model, api_python=api_python)

def call_judge(prompt, context):
    print(f'  {current_note_id}: judge request', flush=True)
    return condition_utils.call_model(prompt, context, root=root, provider=provider, model=judge_model, api_python=api_python)

predictions_path.parent.mkdir(parents=True, exist_ok=True)
with predictions_path.open('w', encoding='utf-8') as predictions, reviews_path.open('w', encoding='utf-8') as reviews:
    for index, note in enumerate(notes, 1):
        current_note_id = note['id']
        print(f'[{index}/{len(notes)}] {current_note_id}', flush=True)
        result = test03_review.review_note(
            note['text'], catalog, valid_codes, doctor_prompt, judge_prompt,
            call_doctor, call_judge, max_retries,
        )
        predictions.write(json.dumps({'id': current_note_id, 'codes': result['codes']}, ensure_ascii=False) + '\n')
        predictions.flush()
        reviews.write(json.dumps({'id': current_note_id, **result}, ensure_ascii=False) + '\n')
        reviews.flush()
        status = 'approved' if result['approved'] else 'unapproved'
        if not result['complete']:
            status += ', incomplete'
        print(f"  {current_note_id}: {status} after {len(result['attempts'])} attempt(s)", flush=True)
print(f'Predictions: {predictions_path}\nReview history: {reviews_path}')

[1/10] dev-051
  dev-051: doctor request
  dev-051: judge request
  dev-051: doctor request
  dev-051: doctor request
  dev-051: doctor request
  dev-051: judge request
  dev-051: approved after 4 attempt(s)
[2/10] dev-052
  dev-052: doctor request
  dev-052: doctor request
  dev-052: doctor request
  dev-052: judge request
  dev-052: doctor request
  dev-052: unapproved, incomplete after 4 attempt(s)
[3/10] dev-053
  dev-053: doctor request
  dev-053: doctor request
  dev-053: doctor request
  dev-053: judge request
  dev-053: doctor request
  dev-053: unapproved, incomplete after 4 attempt(s)
[4/10] dev-054
  dev-054: doctor request
  dev-054: doctor request
  dev-054: judge request
  dev-054: doctor request
  dev-054: doctor request
  dev-054: unapproved, incomplete after 4 attempt(s)
[5/10] dev-055
  dev-055: doctor request
  dev-055: judge request
  dev-055: doctor request
  dev-055: judge request
  dev-055: doctor request
  dev-055: judge request
  dev-055: doctor request
  dev-0

## Score with `score.py`

The scorer reports top-1, top-3, and weighted points. The reviewer's approval is tracked separately and is not treated as ground truth. Missing predictions count as misses.

In [4]:
subprocess.run([
    sys.executable, str(root / 'score.py'),
    '--gold', str(notes_path), '--pred', str(predictions_path),
    '--mms', str(mms_path), '--per-note', str(per_note_path),
], cwd=root, check=True);

notes 10  missing predictions 0  invalid codes 0  duplicate ids 0
slice A: n=10  top1 0.400  top3 0.400  points 30.0 / 75
SCORE 30.0 / 75


## Inspect per-note scores and review history

The `attempts` field shows model feedback and any invalid or missing-CSV candidates. Unapproved or incomplete results are still scored but clearly marked. The predictions file contains only `id` and `codes`, as in the example format.

In [5]:
with per_note_path.open(encoding='utf-8') as file:
    results = [json.loads(line) for line in file if line.strip()]
with reviews_path.open(encoding='utf-8') as file:
    reviews = {row['id']: row for row in (json.loads(line) for line in file if line.strip())}
rows = []
for result in results:
    review = reviews.get(result['id'])
    rows.append({
        'id': result['id'], 'gold': result['gold'], 'pred': result['pred'],
        'top1': result['top1'], 'top3': result['top3'],
        'judge_approved': review['approved'] if review is not None else None,
        'complete': review.get('complete') if review is not None else None,
        'attempts': review['attempts'] if review is not None else None,
    })
print(f'{len(rows)} notes evaluated; {sum(row["judge_approved"] is True for row in rows)} judge-approved.')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150):
    display(pd.DataFrame(rows))

10 notes evaluated; 1 judge-approved.


,id,gold,pred,top1,top3,judge_approved,complete,attempts
0,dev-051,[5B52],"[5B52, 5B51, 5B50]",True,True,True,True,"[{'attempt': 1, 'codes': ['5B50', '5B52', '5B53'], 'approved': False, 'status': 'rejected', 'feedback': 'The primary code should be reconsidered: ..."
1,dev-052,[BD10],[BD10],True,True,False,False,"[{'attempt': 1, 'codes': ['BD10.2', 'BD10.0', 'BD10.1'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Rank 1 is not a valid MMS..."
2,dev-053,[QA42.0],[QA30],False,False,False,False,"[{'attempt': 1, 'codes': ['JB00.0'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Provide exactly three ranked codes.'}, {'atte..."
3,dev-054,[5A40.0],[5A40.0],True,True,False,False,"[{'attempt': 1, 'codes': ['5A40'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Provide exactly three ranked codes.'}, {'attemp..."
4,dev-055,[BA40.1],"[BA40.1, BA40, BA40.Z]",True,True,False,True,"[{'attempt': 1, 'codes': ['BA40.0', 'BA5Z', 'BA40.Z'], 'approved': False, 'status': 'rejected', 'feedback': 'BA40.0 is unsuitable: the note descri..."
5,dev-056,[JA63.2],"[JA63, 5A40, 5A41]",False,False,False,True,"[{'attempt': 1, 'codes': ['JA63'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Provide exactly three ranked codes.'}, {'attemp..."
6,dev-057,[5A00.2Z],"[5A00.1, 5A00.Z]",False,False,False,False,"[{'attempt': 1, 'codes': ['5A00.1'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Provide exactly three ranked codes.'}, {'atte..."
7,dev-058,[BD90.0],[1B73],False,False,False,False,"[{'attempt': 1, 'codes': ['1B70', '4B00', 'CA02'], 'approved': False, 'status': 'rejected', 'feedback': 'The note supports acute cervical lymphade..."
8,dev-059,[JA62.1],"[GC00.0, GC00.Z, GC08.Z]",False,False,False,True,"[{'attempt': 1, 'codes': ['GC08.0', 'GB51.0', 'GC08.Z'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Rank 2 is not a valid MMS..."
9,dev-060,[5B81.01],"[5B81.0, 5B81]",False,False,False,False,"[{'attempt': 1, 'codes': ['5B81'], 'approved': False, 'status': 'invalid_candidates', 'feedback': 'Provide exactly three ranked codes.'}, {'attemp..."
